##Quantizing fine-tuning Llama with Hugging Face dataset
### Model size - 7B
### finetune - PEFT- QLoRa

In [ ]:
!pip install -q accelerate peft bitsandbytes transformers trl datasets torch

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

###Prepare Custom Dataset

In [ ]:
from datasets import load_dataset
dataset_name = "HFS26/custom_reuters_articles" ## dataset
dataset = load_dataset(dataset_name, split="train[:500]") # Load a subset

# formatting function
def format_dataset(example):
    # Adjust based on your specific dataset structure
    #text = f"### Instruction:\\n{example['instruction']}\\n\\n### Response:\\n{example['output']}"
    text = f"### Instruction:\n{example.get('instruction', '')}\n\n### Response:\n{example.get('output', '')}"
    return {"text": text}

dataset = dataset.map(format_dataset)


###Load the Model with 4-bit Quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_name = "meta-llama/Llama-2-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)


###Setup Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


###Set up QLoRA Configuration

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=64, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)


###Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./results",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        num_train_epochs=1,
    ),
    dataset_text_field="text",
    tokenizer=tokenizer,
)
trainer.train()

###Save model

In [ ]:
trainer.push_to_hub()